# Monte-Carlo Payoffs

Author: Sebastien Gurrieri, sebgur@gmail.com

This notebook illustrates the pricing of a number of payoffs by Monte-Carlo, using the payoff library.

In [ ]:
import datetime as dt
from sdevpy.utilities.book import Book
import sdevpy as sd
from sdevpy import pricingcontext
from sdevpy.montecarlo.payoffs import cashflows as cfl
from sdevpy.montecarlo.payoffs.basic import Trade, Instrument, Variance, Sqrt
from sdevpy.montecarlo.payoffs.vanillas import make_vanilla_option
from sdevpy.montecarlo.payoffs.exotics import WorstOfBarrier, make_basket_option, make_asian_option
from sdevpy.montecarlo import mcpricer as mc


print("SDevPy version: " + sd.__version__)
ctx = pricingcontext.default_pricing_context()

SDevPy version: 1.1


### Setup Date and MarketDataProvider
We assume here that all market data is available and all required models have been calibrated.

In [2]:
valdate = dt.datetime(2025, 12, 15)

# Retrieve market and calibration data providers
md_prov = ctx.market_provider
cal_prov = ctx.calib_provider

### Setup Portfolio
In this section we create a few trades, showing example of various ways instruments can be created. The vanilla ones will later be used for comparison against closed-forms.

In [19]:
# Create portfolio
book = Book()
trades = []
start_date = dt.datetime(2025, 11, 15)
expiry = dt.datetime(2026, 12, 15)
expiry2 = dt.datetime(2027, 12, 15)
v_name, v_strike, v_type = 'ABC', 100.0, 'Call' # For check against CF

# Vanilla
index = make_vanilla_option(v_name, v_strike, v_type, expiry)
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Basket option
index = make_basket_option(['XYZ', 'KLM'], [0.5, 0.8], 100.0, 'Call', expiry)
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Asian option
index = make_asian_option('ABC', 100.0, 'Call', valdate, expiry, freq='5D')
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Worst-of barrier
index = WorstOfBarrier(['ABC', 'XYZ'], expiry, 60.0, 'Call', 35.0)
cf = cfl.Cashflow(index, expiry)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# VarSwap
index = Variance('ABC', start_date, expiry)
vstrike = 14.0
payoff = index - vstrike * vstrike
notional = 100.0 * 1000
vega_notional = notional / (2.0 * vstrike)
cf = cfl.Cashflow(payoff, expiry, notional=vega_notional)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# VolSwap
var_index = Variance('ABC', start_date, expiry)
index = Sqrt(var_index)
vstrike = 14.0
payoff = index - vstrike
notional = 100.0 * 1000
vega_notional = notional
cf = cfl.Cashflow(payoff, expiry, notional=vega_notional)
trades.append(Trade(Instrument(cashflow_legs=[[cf]])))

# Create book
book.add_trades(trades)

### Launch the MC simulation
The simulation is triggered with a number of optional parameters related to the path construction.

In [20]:
# Path parameters
constr_type = 'brownianbridge' # brownianbridge, incremental
rng_type = 'sobol' # sobol, mt, halton, latinhypercube
n_paths = 1 * 1000
n_steps = 50

# Price book
mc_price = mc.price_book(valdate, book, ctx, constr_type=constr_type, rng_type=rng_type,
                         n_paths=n_paths, n_timesteps=n_steps)

# Display prices
for i in range(len(mc_price['pv'])):
    print("MC:", mc_price['pv'][i])

Runtime(Generate spot paths): 0.1s
Runtime(Interpolate to event grid): 0.0s
Runtime(Payoff calculation): 0.0s
MC: 11.877720807010398
MC: 13.141772764105912
MC: 6.836053588174807
MC: 2.4451813559247135
MC: 155093.79792883986
MC: 134821.0590649871
